<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_15%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## [문제 1]

강화학습 에이전트는 학습 과정에서 이미 알고 있는 보상이 높은 행동을 선택할지(Exploitation), 아니면 더 높은 보상을 찾기 위해 새로운 행동을 시도할지(Exploration) 결정해야 합니다. 이 딜레마를 해결하기 위해, 초기에는 무작위 행동을 많이 하다가 점차 최적의 행동을 선택하는 비율을 높이는 가장 대표적인 행동 선택 전략(Policy)의 이름은 무엇입니까?

**[답안 작성란]:**엡실론 그리디 정책



## [문제 2]

Deep Q-Network(DQN)에서는 에이전트가 경험한 데이터 `(state, action, reward, next_state, done)`를 바로 학습에 사용하지 않고, 메모리 버퍼(Buffer)에 저장해 두었다가 무작위로 추출(Sampling)하여 학습합니다. 데이터 간의 상관관계를 끊고 학습을 안정화하기 위한 이 기법의 명칭은 무엇입니까?

**[답안 작성란]:** 경험 재생 (Experience Replay)



## [문제 3]

Q-learning의 핵심은 벨만 최적 방정식을 이용하여 Q-table의 값을 갱신하는 것입니다. 다음 수식을 코드로 구현하시오.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s,a) \right]$$

* `current_q`: $Q(s,a)$
* `max_next_q`: $\max_{a'} Q(s', a')$
* `self.lr`: $\alpha$ (학습률)
* `self.gamma`: $\gamma$ (할인율)

In [2]:
import numpy as np

# 예시 변수 (실제 클래스 내부 메서드라고 가정)
class QLearningAgentSnippet:
    def __init__(self):
        self.lr = 0.1
        self.gamma = 0.99
        self.q_table = np.zeros((16, 4)) # 예: FrozenLake 16 states, 4 actions

    def update(self, state, action, reward, next_state, done):
        # 1. 현재 상태의 Q-value 가져오기
        current_q = self.q_table[state, action]

        # 2. 다음 상태에서의 최대 Q-value 구하기
        # TODO: self.q_table[next_state] 중에서 가장 큰 값을 구하세요.
        max_next_q = np.max(self.q_table[next_state])

        # 3. 타겟(Target) 값 계산
        if done:
            target = reward
        else:
            # TODO: reward + gamma * max_next_q 를 작성하세요.
            target = reward + self.gamma * max_next_q

        # 4. Q-table 업데이트
        # TODO: Q-learning 업데이트 공식을 완성하세요.
        self.q_table[state, action] = current_q + self.lr * (target - current_q)

        return self.q_table[state, action]

## [문제 4]

CartPole 환경(상태 차원: 4, 행동 개수: 2)을 위한 DQN 모델을 PyTorch로 정의하시오.
**요구사항:**
1. 입력층(state_dim) -> 은닉층(128) -> 은닉층(128) -> 출력층(action_dim) 구조를 가질 것.
2. 은닉층 사이에는 `ReLU` 활성화 함수를 사용할 것.

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        # TODO: nn.Linear를 사용하여 3개의 층(fc1, fc2, fc3)을 정의하세요.
        # Hidden unit 크기는 128로 설정합니다.
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        # TODO: fc1 -> relu -> fc2 -> relu -> fc3 순서로 연결하세요.
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## [문제 5]

Experience Replay Buffer에서 무작위로 데이터를 추출하여 학습 가능한 텐서 형태로 변환하는 코드를 작성하시오.

In [4]:
import random
import torch
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        # 1. 버퍼에서 batch_size만큼 랜덤하게 샘플링
        # TODO: random.sample을 사용하세요.
        batch = random.sample(self.buffer, batch_size)

        # 2. 데이터를 텐서로 변환 (리스트 컴프리헨션 사용)
        # batch의 각 요소는 (s, a, r, ns, d) 튜플입니다.
        # TODO: 각 요소를 분리하여 torch.FloatTensor 또는 LongTensor로 변환하세요.
        # 힌트: states는 float, actions는 long(정수)이어야 합니다.
        states = torch.FloatTensor([t[0] for t in batch])
        actions = torch.LongTensor([t[1] for t in batch])
        rewards = torch.FloatTensor([t[2] for t in batch])
        next_states = torch.FloatTensor([t[3] for t in batch])
        dones = torch.FloatTensor([t[4] for t in batch])

        return states, actions, rewards, next_states, dones

## [문제 6]

DQN 학습 단계에서 **Target Q-value**를 계산하고, 현재 Q-value와의 차이(Loss)를 구하는 코드를 작성하시오.

* `gamma`: 할인율
* `target_network`: 타겟 신경망 (고정된 파라미터 사용)
* 공식: $Target = r + \gamma \max_{a'} Q_{target}(s', a') \times (1 - done)$

In [5]:
# 학습 루프 내부라고 가정
# inputs: states, actions, rewards, next_states, dones

def calculate_loss(agent, states, actions, rewards, next_states, dones):
    # 1. 현재 상태의 Q-value (우리가 예측한 값)
    # gather를 사용하여 선택한 행동의 Q-value만 가져옵니다.
    current_q_values = agent.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # 2. 다음 상태의 최대 Q-value (타겟 네트워크 사용)
    # TODO: agent.target_network를 사용하여 next_states의 Q값을 구하고, .max(1)[0]을 취하세요.
    with torch.no_grad():
        max_next_q_values = agent.target_network(next_states).max(1)[0]

        # 3. 벨만 최적 방정식을 이용한 타겟 값 계산
        # TODO: rewards + gamma * max_next_q_values * (1 - dones) 를 구현하세요.
        target_q_values = rewards + agent.gamma * max_next_q_values * (1 - dones)

    # 4. 손실(MSE Loss) 계산
    # TODO: current_q_values와 target_q_values 사이의 MSE Loss를 계산하세요. (nn.MSELoss 사용 가정)
    loss_fn = nn.MSELoss()
    loss = loss_fn(current_q_values, target_q_values)

    return loss